# Masking

You have three things built:

```
Encoder  → H: (batch, 512, L)
Separator → masks: (batch, C, 512, L)    values in [0,1]
Decoder  → takes (batch, 512, L), returns waveform   ← next session
```

The masking step sits between separator and decoder:

```
H:    (batch, 512, L)        one shared representation of the mixture
mask: (batch, C, 512, L)     C speakers, each with 512 filter weights

masked = H.unsqueeze(1) * mask   → (batch, C, 512, L)
```

`H.unsqueeze(1)` adds a speaker dimension so broadcasting works — every speaker's mask multiplies against the same `H`. The result is `C` separate representations, each emphasizing the encoder features that belong to that speaker.

Concretely with numbers:

```
H[0, 42, 100] = 0.8      ← filter 42 at frame 100 was strongly activated

mask[0, 0, 42, 100] = 0.91   ← separator says: speaker 0 owns 91% of this
mask[0, 1, 42, 100] = 0.12   ← separator says: speaker 1 owns 12% of this

masked[0, 0, 42, 100] = 0.8 × 0.91 = 0.73   ← speaker 0's version
masked[0, 1, 42, 100] = 0.8 × 0.12 = 0.10   ← speaker 1's version
```

Notice the masks don't need to sum to 1.0 across speakers. Each is independently in `[0,1]` — the sigmoid ensures this. The network learns to push masks toward 0 or 1 through training.

Now you can also see why the `1×1 Conv + Sigmoid` at the end of the separator matters. Without sigmoid, the mask values could be anything — negative, greater than 1 — and the masking multiplication would produce nonsensical results. Sigmoid squashes everything to `[0,1]`, making masks interpretable as "how much of this filter's energy belongs to this speaker."


## Session 3.3: Decoder Theory

### The reconstruction problem

After masking you have `(batch, C, N, L)` — each speaker's representation in the learned encoder space. The decoder's job: invert the encoder transformation and recover a time-domain waveform.

The encoder did this:

```
raw audio → slide kernel of size 16, step 8 → latent representation
```

Think of what that means physically. For every 8 samples in the original audio, the encoder produced one frame of `N=512` values. You threw away a lot of information in that compression — or rather, you *transformed* it into a different representational space.

The decoder needs to go backwards:

```
latent representation → reconstruct time-domain samples
```

### Transposed convolution — what it actually does

A regular `Conv1d` with stride `S` takes one step of size `S` in the input for each output position. A `ConvTranspose1d` with stride `S` does the opposite — it takes one input position and produces `S` output positions.

Concretely, for one filter and stride=8:

```
Regular Conv1d (stride=8):
  input positions:   [0,  8, 16, 24, ...]   ← steps of 8
  output positions:  [0,  1,  2,  3, ...]   ← one output per step

ConvTranspose1d (stride=8):
  input positions:   [0,  1,  2,  3, ...]   ← every frame
  output positions:  [0,  8, 16, 24, ...]   ← each frame contributes to 8 output samples
```

Each latent frame "spreads" its contribution across `kernel_size=16` output samples, stepping by `stride=8`. Adjacent frames' contributions overlap by `kernel_size - stride = 16 - 8 = 8` samples. The final output at each sample position is the *sum* of all overlapping contributions — this is the **overlap-add** reconstruction you saw in the roadmap.

Here's what that overlap looks like:

```
Frame 0 contributes to output samples:  0 → 15
Frame 1 contributes to output samples:  8 → 23
Frame 2 contributes to output samples:  16 → 31

Samples 8-15 receive contributions from BOTH frame 0 and frame 1
Samples 16-23 receive contributions from BOTH frame 1 and frame 2
```

The 50% overlap (stride=8, kernel=16) means every output sample gets contribution from exactly 2 frames. This is not an accident — it's why the encoder uses 50% overlap. The decoder can perfectly reconstruct smooth audio because no sample is "orphaned" — every sample gets multiple overlapping contributions that smooth out discontinuities between frames.

### Is this a true inverse?

Here's something important: `ConvTranspose1d` is **not** a mathematical inverse of `Conv1d`. It's a learnable pseudo-inverse. Before any training, if you pass audio through encoder → decoder, you get noise — the random weights produce meaningless output.

After training, the decoder weights are optimized to produce the best possible waveform given the masked representation. The encoder simultaneously learns a representation that the decoder can reconstruct from. They co-adapt. This is fundamentally different from STFT, where the inverse (ISTFT) is mathematically exact.

### Phase recovery — why time-domain matters

In STFT-based systems, the spectrogram discards phase information. Reconstructing audio from a magnitude spectrogram requires *estimating* the phase — a hard problem that introduces artifacts. This is why classical vocoder-based speech synthesis sounds robotic.

Conv-TasNet sidesteps this entirely by working in the time domain. The encoder captures both amplitude and phase implicitly in its learned representation. The decoder reconstructs both simultaneously. No phase estimation needed — it's one of the key advantages of the time-domain approach, and why Conv-TasNet outperformed earlier STFT-based separation systems significantly.

---

## TODO 4: Implement the Decoder

**Goal:** Build a `Decoder` class that converts a masked latent representation back to a waveform.

**Requirements:**
- Takes `(batch, N, L)` as input — one speaker's masked representation
- Returns `(batch, 1, T)` — reconstructed waveform
- Uses `nn.ConvTranspose1d` with same `num_filters`, `kernel_size`, `stride` as encoder
- No activation — raw waveform output, values can be freely positive or negative
- Default parameters match paper: `num_filters=512, kernel_size=16, stride=8`

**Architecture:**
```
input: (batch, N, L)
  → ConvTranspose1d(in_channels=N, out_channels=1, kernel_size=16, stride=8, bias=False)
output: (batch, 1, T_approx)
```

**Hints:**
- `in_channels` and `out_channels` are swapped vs the encoder — encoder went `1 → N`, decoder goes `N → 1`
- Output length formula for ConvTranspose1d: `T_out = (L - 1) * stride + kernel_size`
- Verify: `(59999 - 1) * 8 + 16 = ?` — is that close to your original 480000 samples?
- Don't worry about exact length matching yet — just verify the shape is `(batch, 1, ~T)`

**Expected:**
```python
decoder = Decoder(num_filters=512, kernel_size=16, stride=8)
d = torch.randn(4, 512, 59999)
out = decoder(d)
print(out.shape)   # (4, 1, 480008) — close to but not exactly 480000
```

**After it works — two questions:**

1. You get `480008` instead of `480000`. Where do those extra 8 samples come from? Work it out using the formula above.

2. The encoder used `bias=False` and the decoder should too. Why is `bias=False` conventional for both? What would a learned bias do to the reconstructed waveform?

Go implement, come back with code and answers to both questions.